# Target-Conditioned Peptide Binder Design
 Generative peptide design (PepMLM) + co-folding/"docking" (Boltz-1) + structural superposition + statistical analysis + reporting

## What you will build

1. **Targets** — two given targets, plus **one you find yourself** using the RCSB Search API against criteria you're given.
2. **Candidate generation with PepMLM**, conditioned on each target — *and* a **composition-matched scrambled control** for every candidate, so later results can be judged against a null distribution instead of taken at face value.
3. **Co-folding with Boltz-1** for every (target, real-or-scrambled peptide) pair.
4. **Structural superposition**: align each predicted complex's target chain onto the *original crystal structure* and measure how much the predicted peptide pose actually overlaps the real, experimentally observed peptide footprint. Sequences and residue numbering will not match up trivially — that mismatch is itself part of the exercise.
5. **Per-residue confidence mapped onto the B-factor column**, so VMD can color the structure by model confidence rather than only by chain.
6. **Statistics, not vibes**: is the real-peptide confidence distribution actually different from the scrambled-control distribution (Mann–Whitney U)? Does the language model's own confidence predict the structure model's confidence (Spearman correlation, with a p-value)? You design and justify a composite ranking score.
7. **A report** that has to defend its conclusions using the above, not just present a sorted table.

## No training required
Every model is pretrained and frozen; this is an inference + analysis exercise. The difficulty here is in *correctly wiring tools together and interpreting their output critically* — including catching cases where the metrics disagree with each other.

## Pipeline overview

```
 2 given targets + 1 target you find yourself (RCSB Search API)
          │
          ▼
   [1] PepMLM  ──►  N real candidate peptides per target
          │                     │
          │            composition-matched scrambled controls
          ▼                     ▼
   [2] Boltz-1 co-folding for EVERY (target, peptide) pair, real + scrambled
          │
          ▼
   [3] Parse confidence JSON + write per-residue confidence into B-factor column
          │
          ▼
   [4] Superimpose predicted target chain onto the ORIGINAL crystal structure;
       measure predicted-peptide overlap with the real peptide's footprint
          │
          ▼
   [5] Statistics: real vs scrambled (Mann-Whitney U), avg_log_prob vs iptm
       (Spearman), your own justified composite ranking score
          │
          ▼
   [6] Report defending a ranked shortlist using the evidence above
```

> **Submission requirement:** Please send the notebook file and the final report as PDF.

## Step 0 — Environment Setup

In [ ]:
import torch
import sys

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: c:\Users\user\.venv\Scripts\python.exe
Torch: 2.13.0+cpu
CUDA build: None
CUDA available: False


: 

In [3]:
%pip uninstall -y numpy pandas boltz

%pip install -q torch transformers accelerate
%pip install -q "numpy<2.0.0" pandas matplotlib scipy biopython py3Dmol pyyaml requests
%pip install -q boltz

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 3.0.5
Uninstalling pandas-3.0.5:
  Successfully uninstalled pandas-3.0.5
Found existing installation: boltz 2.2.1
Uninstalling boltz-2.2.1:
  Successfully uninstalled boltz-2.2.1
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
biotite 0.41.2 requires numpy<2.0,>=1.14.5, but you have numpy 2.4.6 which is incompatible.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.4.6 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.4.6 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Environment setup — run once.
%pip install -q torch transformers accelerate
%pip install -q boltz -U
%pip install -q biopython py3Dmol pandas matplotlib pyyaml requests scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os, json, random, subprocess, itertools, textwrap
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml
from scipy import stats

WORKDIR = Path("peptide_binder_practice")
(WORKDIR / "boltz_inputs").mkdir(parents=True, exist_ok=True)
(WORKDIR / "boltz_outputs").mkdir(parents=True, exist_ok=True)
(WORKDIR / "vmd_scripts").mkdir(parents=True, exist_ok=True)
(WORKDIR / "reference_structures").mkdir(parents=True, exist_ok=True)

RNG = random.Random(0)
print("Working directory:", WORKDIR.resolve())


Working directory: C:\Users\user\Desktop\Term6\computational drug design\peptide_binder_practice


## Step 1 — Targets: Two Given, One You Find

Two given anchors:

| Target | PDB ID | Chain(s) | Known binder |
|---|---|---|---|
| MDM2 (N-terminal domain) | `1YCR` | A (target), B (p53 peptide) | p53 transactivation-domain peptide |
| Bcl-xL | `1BXL` | A (target), B (BAK BH3 peptide) | BAK BH3 peptide |

**Your job:** find a **third** suitable target yourself using the [RCSB Search API](https://search.rcsb.org/#search-api) (`POST https://search.rcsb.org/rcsbsearch/v2/query`), subject to these constraints:
- Exactly two polymer entities (a target chain + a short peptide chain) — i.e. a genuine binary protein–peptide complex, not a larger assembly.
- Target chain roughly 60–300 residues; peptide chain roughly 8–30 residues.
- Solved by X-ray crystallography (so both chains have real coordinates you can later use as ground truth for the superposition step).

Note that RCSB's search API schema is not something to guess blindly — consult the docs linked above (the "full-text" and "structure attribute" search services are the most useful starting points) and iterate on your query until it returns sensible hits. You do not need to fully understand every RCSB search feature — a working query using 2–3 filters is enough.

Once you have all three PDB IDs, reuse the FASTA-fetching approach to pull `target_sequence` and `reference_peptide_sequence` for each, exactly as you would for the two given targets.

In [6]:
import requests


TARGETS = {
    "MDM2": {
        "pdb_id": "1YCR",
        "target_chain": "A",
        "reference_peptide_chain": "B",
        "n_peptides": 4,
        "peptide_length_range": (12, 18),
    },
    "BCL_XL": {
        "pdb_id": "1BXL",
        "target_chain": "A",
        "reference_peptide_chain": "B",
        "n_peptides": 4,
        "peptide_length_range": (14, 20),
    },
}


def search_rcsb_for_binary_peptide_complex(
    min_target_len=60,
    max_target_len=300,
    min_peptide_len=8,
    max_peptide_len=30,
    rows=25,
) -> list:
    """
    Query the RCSB Search API and return candidate PDB IDs
    for plausible binary target+peptide complexes.
    """

    url = "https://search.rcsb.org/rcsbsearch/v2/query"

    query = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.polymer_entity_count",
                        "operator": "equals",
                        "value": 2,
                    },
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "exptl.method",
                        "operator": "exact_match",
                        "value": "X-RAY DIFFRACTION",
                    },
                },
            ],
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": rows,
            }
        },
    }

    response = requests.post(url, json=query)
    response.raise_for_status()

    data = response.json()

    return [item["identifier"] for item in data["result_set"]]


def fetch_pdb_fasta(pdb_id: str) -> dict:
    """
    Return {chain_id: sequence} for every polymer chain in a PDB entry.
    """

    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/download"

    response = requests.get(url)
    response.raise_for_status()

    chains = {}

    header = None
    sequence = []

    for line in response.text.splitlines():

        if line.startswith(">"):

            if header is not None:
                chain_id = header.split("|")[1].replace("Chain ", "").strip()
                chains[chain_id] = "".join(sequence)

            header = line
            sequence = []

        else:
            sequence.append(line.strip())

    if header is not None:
        chain_id = header.split("|")[1].replace("Chain ", "").strip()
        chains[chain_id] = "".join(sequence)

    return chains


# 1. Get candidate PDB IDs

candidate_pdb_ids = search_rcsb_for_binary_peptide_complex()


# 2. Find one candidate with one target-length chain
#    and one peptide-length chain

for pdb_id in candidate_pdb_ids:

    if pdb_id in ["1YCR", "1BXL"]:
        continue

    chains = fetch_pdb_fasta(pdb_id)

    target_chains = [
        chain
        for chain, seq in chains.items()
        if 60 <= len(seq) <= 300
    ]

    peptide_chains = [
        chain
        for chain, seq in chains.items()
        if 8 <= len(seq) <= 30
    ]

    if len(target_chains) >= 1 and len(peptide_chains) >= 1:

        target_chain = target_chains[0]
        peptide_chain = peptide_chains[0]

        if target_chain != peptide_chain:

            # 3. Add it to TARGETS

            TARGETS["FOUND_TARGET"] = {
                "pdb_id": pdb_id,
                "target_chain": target_chain,
                "reference_peptide_chain": peptide_chain,
                "n_peptides": 4,
                "peptide_length_range": (8, 30),
            }

            break


# 4. Fetch sequences for all three targets

for name, info in TARGETS.items():

    chains = fetch_pdb_fasta(info["pdb_id"])

    info["target_sequence"] = chains[info["target_chain"]]

    info["reference_peptide_sequence"] = chains[
        info["reference_peptide_chain"]
    ]


for name, info in TARGETS.items():

    print(
        name,
        info.get("pdb_id"),
        "-- sequences not yet fetched"
        if "target_sequence" not in info
        else "ok",
    )

MDM2 1YCR ok
BCL_XL 1BXL ok
FOUND_TARGET 10KU ok


## Step 2 — Candidate Generation with PepMLM, Plus a Negative Control

Use **PepMLM** (`TianlaiChen/PepMLM-650M`, Hugging Face) to propose peptides conditioned on each target sequence, as before. This time the requirements are stated at a higher level — decide your own decoding strategy (one-shot fill vs. iterative unmasking; greedy vs. temperature-sampled) and be ready to justify the choice in your report.

Required outputs per generated peptide:
- `sequence` — the amino-acid string.
- `avg_log_prob` — a generation-confidence score you define (e.g. average log-probability of the chosen tokens under the model). State exactly how you computed it.

**Negative control (new requirement):** for every real generated peptide, also produce a **composition-matched scrambled peptide** — i.e. a random permutation of the *same* amino acids (so amino-acid composition is identical, only order differs). This tests whether Boltz-1's downstream confidence is actually sensitive to *sequence order/identity conditioned on the target*, or whether it would score any peptide of similar composition just as well. Store both sets in one DataFrame with a `source` column (`"pepmlm"` or `"scrambled"`), and give scrambled peptides `avg_log_prob = NaN` (the language-model score doesn't apply to a shuffled sequence).

In [7]:
import torch
print(torch.__version__)

2.13.0+cpu


In [6]:

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM


PEPMLM_MODEL_ID = "ChatterjeeLab/PepMLM-650M"

device = "cuda" if torch.cuda.is_available() else "cpu"

pepmlm_tokenizer = AutoTokenizer.from_pretrained(PEPMLM_MODEL_ID)

pepmlm_model = (
    AutoModelForMaskedLM
    .from_pretrained(PEPMLM_MODEL_ID)
    .to(device)
    .eval()
)


def generate_peptide(
    target_sequence: str,
    peptide_length: int,
    temperature: float = 1.0
) -> tuple[str, float]:
    """
    Generate one peptide for target_sequence using one-shot
    temperature sampling.

    avg_log_prob is the mean log-probability of the sampled
    amino acids at the masked peptide positions.
    """

    masked_sequence = (
        target_sequence
        + pepmlm_tokenizer.mask_token * peptide_length
    )

    inputs = pepmlm_tokenizer(
        masked_sequence,
        return_tensors="pt"
    ).to(device)

    mask_positions = (
        inputs["input_ids"][0]
        == pepmlm_tokenizer.mask_token_id
    )

    with torch.no_grad():
        logits = pepmlm_model(**inputs).logits[0, mask_positions]

    amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

    aa_token_ids = torch.tensor(
        [
            pepmlm_tokenizer.convert_tokens_to_ids(aa)
            for aa in amino_acids
        ],
        device=device
    )

    aa_logits = logits[:, aa_token_ids] / temperature

    probabilities = torch.softmax(
        aa_logits,
        dim=-1
    )

    sampled_indices = torch.multinomial(
        probabilities,
        num_samples=1
    ).squeeze(-1)

    peptide = "".join(
        amino_acids[i]
        for i in sampled_indices.tolist()
    )

    log_probs = torch.log(
        probabilities.gather(
            1,
            sampled_indices.unsqueeze(1)
        ).squeeze(1)
    )

    avg_log_prob = log_probs.mean().item()

    return peptide, avg_log_prob


def scramble_peptide(
    sequence: str,
    rng: random.Random
) -> str:
    """
    Return a composition-matched scrambled peptide.
    """

    residues = list(sequence)

    rng.shuffle(residues)

    return "".join(residues)


rows = []

rng = random.Random(42)

temperature = 1.0


# Generate PepMLM peptides and scrambled controls

for target_name, info in TARGETS.items():

    min_len, max_len = info["peptide_length_range"]

    for i in range(info["n_peptides"]):

        peptide_length = rng.randint(
            min_len,
            max_len
        )

        sequence, avg_log_prob = generate_peptide(
            info["target_sequence"],
            peptide_length,
            temperature
        )

        peptide_id = f"{target_name}_{i + 1}"

        rows.append({
            "target": target_name,
            "peptide_id": peptide_id,
            "source": "pepmlm",
            "sequence": sequence,
            "length": len(sequence),
            "temperature": temperature,
            "avg_log_prob": avg_log_prob,
        })

        scrambled_sequence = scramble_peptide(
            sequence,
            rng
        )

        rows.append({
            "target": target_name,
            "peptide_id": peptide_id,
            "source": "scrambled",
            "sequence": scrambled_sequence,
            "length": len(scrambled_sequence),
            "temperature": temperature,
            "avg_log_prob": np.nan,
        })


peptides_df = pd.DataFrame(rows)

peptides_df.to_csv(
    WORKDIR / "generated_peptides.csv",
    index=False
)

peptides_df

c:\Users\user\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\user\.venv\Lib\site-packages\huggingface_hub\file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 2609.50 MB. The target location C:\Users\user\.cache\huggingface\hub\models--ChatterjeeLab--PepMLM-650M\blobs only has 1682.51 MB free disk space.
  warnings.warn(


,target,peptide_id,source,sequence,length,temperature,avg_log_prob
0,MDM2,MDM2_1,pepmlm,QSMEVRETFEYYWQLLP,17,1.0,-1.414025
1,MDM2,MDM2_1,scrambled,TYRELLQEFSMWPVYQE,17,1.0,NaN
2,MDM2,MDM2_2,pepmlm,TTFFYYRKLLMCT,13,1.0,-1.762363
3,MDM2,MDM2_2,scrambled,KYMFYCTRFTTLL,13,1.0,NaN
4,MDM2,MDM2_3,pepmlm,TSNFYPEFSEEYWDRLS,17,1.0,-1.866479
5,MDM2,MDM2_3,scrambled,WFTYRSSEELSPFNYED,17,1.0,NaN
6,MDM2,MDM2_4,pepmlm,QSQFTPTFKLYWCNLE,16,1.0,-1.836109
7,MDM2,MDM2_4,scrambled,QYFCNQWEPLTKSTLF,16,1.0,NaN
8,BCL_XL,BCL_XL_1,pepmlm,MEIDGALLRISVGDLNDQIL,20,1.0,-2.410834
9,BCL_XL,BCL_XL_1,scrambled,DLVEMLIGINQIADSLRGDL,20,1.0,NaN


## Step 3 — Boltz-1 Inputs for Every (Target, Peptide) Pair

Same YAML schema as a standard two-protein Boltz-1 complex (look up the current schema in the Boltz repo/docs if you don't already have it from a previous exercise — chain A gets the target sequence with a real MSA, chain B gets the peptide sequence with `msa: empty`).

The only new requirement: you now have **twice as many peptides** (real + scrambled) per target, so make sure your naming scheme (e.g. `{target}_{peptide_id}_{source}.yaml`) keeps every complex's inputs unambiguous and easy to trace back to a row of `peptides_df` later.

In [7]:
from pathlib import Path


def write_boltz_yaml(
    target_name: str,
    target_sequence: str,
    peptide_id: int,
    source: str,
    peptide_sequence: str,
    out_dir: Path
) -> Path:
    """
    Write a Boltz-1 YAML config for this (target, peptide) complex,
    named f"{target_name}_{peptide_id}_{source}.yaml",
    and return the path.
    """

    out_dir.mkdir(parents=True, exist_ok=True)

    output_path = out_dir / f"{target_name}_{peptide_id}_{source}.yaml"

    yaml_text = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {target_sequence}
  - protein:
      id: B
      sequence: {peptide_sequence}
      msa: empty
"""

    output_path.write_text(yaml_text)

    return output_path


boltz_input_dir = WORKDIR / "boltz_inputs"
written = []


# Write one YAML file for every row in peptides_df

for _, row in peptides_df.iterrows():

    target_name = row["target"]

    path = write_boltz_yaml(
        target_name=target_name,
        target_sequence=TARGETS[target_name]["target_sequence"],
        peptide_id=row["peptide_id"],
        source=row["source"],
        peptide_sequence=row["sequence"],
        out_dir=boltz_input_dir,
    )

    written.append(path)


print(f"Wrote {len(written)} Boltz-1 input configs.")

Wrote 24 Boltz-1 input configs.


## Step 4 — Run Boltz-1

Same CLI pattern as before (`boltz predict <input_dir> --out_dir <out_dir> --use_msa_server --output_format pdb`), pointed at the full `boltz_inputs` directory (now containing real + scrambled complexes for all three targets). No new concept here — the requirement is just to run it and confirm every expected output was produced, since with 2x the complexes, partial failures are more likely to slip past unnoticed.

In [1]:
import subprocess

In [2]:


boltz_out_dir = WORKDIR / "boltz_outputs"

command = [
    "boltz",
    "predict",
    str(boltz_input_dir),
    "--out_dir",
    str(boltz_out_dir),
    "--use_msa_server",
    "--output_format",
    "pdb",
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

NameError: name 'WORKDIR' is not defined

## Step 5 — Parse Outputs, and Extract Per-Residue Confidence

As before: load each complex's predicted PDB and confidence JSON, pull out whatever complex-level fields you'll use (`iptm` etc — inspect `data.keys()` yourself first, don't assume the schema).

**New requirement:** also extract a **per-residue confidence array** from the JSON (Boltz-1 exposes some form of per-residue/per-token confidence — the exact key name and array length/ordering are for you to determine by inspection; it should have one value per residue across both chains, or you may need to compute/derive an equivalent). Then **write those values into the B-factor column** of a copy of the predicted PDB file using Biopython, so the structure can later be colored by confidence in VMD (`mol color Beta`) instead of only by chain. Save these "confidence-colored" PDBs to a separate subfolder so you don't overwrite the originals.

In [ ]:
import json
import warnings
import numpy as np
from Bio.PDB import PDBParser, PDBIO


def find_complex_outputs(name: str, boltz_out_dir: Path) -> dict:
    """
    Locate the predicted PDB, confidence JSON, and per-residue pLDDT
    file for complex `name`.
    """

    pred_dir = boltz_out_dir / "predictions" / name

    if not pred_dir.exists():
        return {}

    pdb_files = list(pred_dir.glob("*_model_0.pdb"))
    confidence_files = list(pred_dir.glob("confidence_*_model_0.json"))
    plddt_files = list(pred_dir.glob("plddt_*_model_0.npz"))

    if not pdb_files or not confidence_files or not plddt_files:
        return {}

    return {
        "pdb_path": pdb_files[0],
        "confidence_json": confidence_files[0],
        "plddt_path": plddt_files[0],
    }


def parse_confidence(json_path: Path, plddt_path: Path) -> dict:
    """
    Load complex-level confidence from JSON and per-residue
    confidence from the pLDDT NPZ file.
    """

    with open(json_path, "r") as f:
        data = json.load(f)

    print("Confidence keys:", data.keys())

    plddt_data = np.load(plddt_path)

    per_residue_confidence = plddt_data["plddt"]

    return {
        "confidence_score": data.get("confidence_score"),
        "ptm": data.get("ptm"),
        "iptm": data.get("iptm"),
        "complex_plddt": data.get("complex_plddt"),
        "complex_iplddt": data.get("complex_iplddt"),
        "per_residue_confidence": per_residue_confidence,
    }


def write_bfactor_pdb(
    pdb_path: Path,
    per_residue_values,
    out_path: Path
):
    """
    Write a copy of the PDB with each atom's B-factor set to
    its residue-level confidence value.
    """

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prediction", pdb_path)

    residues = [
        residue
        for residue in structure.get_residues()
        if residue.id[0] == " "
    ]

    if len(residues) != len(per_residue_values):
        warnings.warn(
            f"Skipping {pdb_path.name}: "
            f"{len(residues)} residues in PDB but "
            f"{len(per_residue_values)} confidence values."
        )
        return None

    for residue, confidence in zip(
        residues,
        per_residue_values
    ):
        for atom in residue.get_atoms():
            atom.set_bfactor(float(confidence))

    io = PDBIO()
    io.set_structure(structure)
    io.save(str(out_path))

    return out_path


bfactor_dir = WORKDIR / "confidence_colored_pdbs"
bfactor_dir.mkdir(exist_ok=True)

results_rows = []


for _, row in peptides_df.iterrows():

    name = f"{row['target']}_{row['peptide_id']}_{row['source']}"

    try:
        outputs = find_complex_outputs(
            name,
            boltz_out_dir
        )

        if not outputs:
            print(f"Missing outputs for {name}")
            continue

        confidence = parse_confidence(
            outputs["confidence_json"],
            outputs["plddt_path"]
        )

        bfactor_path = (
            bfactor_dir /
            f"{name}_confidence.pdb"
        )

        written_path = write_bfactor_pdb(
            outputs["pdb_path"],
            confidence["per_residue_confidence"],
            bfactor_path
        )

        result_row = row.to_dict()

        result_row["pdb_path"] = str(
            outputs["pdb_path"]
        )

        result_row["bfactor_pdb_path"] = (
            str(written_path)
            if written_path is not None
            else None
        )

        for key, value in confidence.items():
            if key != "per_residue_confidence":
                result_row[key] = value

        results_rows.append(result_row)

    except Exception as e:
        print(f"Failed for {name}: {e}")


results_df = pd.DataFrame(results_rows)

results_df.to_csv(
    WORKDIR / "boltz_results.csv",
    index=False
)

results_df.head()

## Step 6 — Visualization: Confidence-Colored VMD Scenes

> ⚠️ **Optional step — you can skip this and come back later.** Step 6 is purely for visual inspection; nothing here feeds into Step 7 or Step 8, which are computed directly from the PDB coordinates and confidence JSON via Biopython/scipy, independently of VMD or py3Dmol. If you're short on time, skip ahead to Step 7 now and only return here afterward to visualize a handful of specific complexes — e.g. your top-ranked ones from Step 8, plus one low-ranked one for contrast. On Kaggle, VMD itself isn't available in the notebook — use `py3Dmol` for a quick in-notebook look, and download the generated `.tcl` + confidence-colored `.pdb` files to open in VMD locally for a higher-quality render.

Generate one VMD `.tcl` script per complex that:
- Shows chain A (target) as cartoon and chain B (peptide) as licorice, as before.
- Colors **by B-factor** (`mol color Beta`) instead of by chain, using the confidence-colored PDB from Step 5 — so low-confidence regions visually stand out.
- Still highlights the interface (chain A residues within 5 Å of chain B).

Also keep a `py3Dmol` viewer for a quick in-notebook look, now colored by the `b` property (py3Dmol supports `{"cartoon": {"colorscheme": {"prop": "b", "gradient": ..., "min": ..., "max": ...}}}`-style styling — check the py3Dmol docs for the exact option names).

In [ ]:
import py3Dmol


VMD_TEMPLATE = """mol new {pdb_filename}
mol delrep 0 top

mol representation NewCartoon
mol color Beta
mol selection "chain A"
mol addrep top

mol representation Licorice
mol color Beta
mol selection "chain B"
mol addrep top

mol representation Surface
mol color ColorID 1
mol selection "chain A and same residue as (within 5 of chain B)"
mol addrep top

display resetview
render TachyonInternal {name}_render.tga
"""


def write_vmd_script(
    bfactor_pdb_path: Path,
    out_dir: Path
) -> Path:
    """
    Fill VMD_TEMPLATE for this confidence-colored complex
    and write the .tcl script; return the path.
    """

    out_dir.mkdir(parents=True, exist_ok=True)

    name = bfactor_pdb_path.stem

    script_text = VMD_TEMPLATE.format(
        pdb_filename=bfactor_pdb_path.resolve(),
        name=name
    )

    out_path = out_dir / f"{name}.tcl"

    out_path.write_text(script_text)

    return out_path


vmd_script_dir = WORKDIR / "vmd_scripts"
vmd_script_dir.mkdir(exist_ok=True)


# Write a VMD script for every available confidence-colored PDB

for pdb_path in results_df["bfactor_pdb_path"].dropna():

    write_vmd_script(
        Path(pdb_path),
        vmd_script_dir
    )


def view_complex_by_confidence(
    pdb_path: str,
    width=500,
    height=400
):
    """
    py3Dmol view of the complex colored by B-factor.
    """

    with open(pdb_path, "r") as f:
        pdb_data = f.read()

    viewer = py3Dmol.view(
        width=width,
        height=height
    )

    viewer.addModel(
        pdb_data,
        "pdb"
    )

    confidence_colors = {
        "prop": "b",
        "gradient": "roygb",
        "min": 0,
        "max": 1
    }

    # Target chain
    viewer.setStyle(
        {"chain": "A"},
        {
            "cartoon": {
                "colorscheme": confidence_colors
            }
        }
    )

    # Peptide chain
    viewer.setStyle(
        {"chain": "B"},
        {
            "stick": {
                "colorscheme": confidence_colors
            }
        }
    )

    viewer.zoomTo()

    return viewer


# Show one example

example_path = (
    results_df["bfactor_pdb_path"]
    .dropna()
    .iloc[0]
)

view_complex_by_confidence(
    example_path
).show()

## Step 7 — Structural Superposition: Does the Predicted Pose Match Reality?

This is the core hard step. For each target, you have a **reference crystal structure** (the original PDB entry) showing exactly where the real peptide binds. You want to know: does Boltz-1's predicted peptide pose land in the same place?

The complication: the predicted complex and the reference crystal structure are two *different* coordinate frames, and the reference structure's chain numbering will often have gaps (unresolved residues) that don't match the predicted structure's residue numbering — even though the underlying target sequence is the same one you gave Boltz-1 as input.

**Required steps (design your own implementation):**
1. Download the original reference structure's coordinate file for each target (`https://files.rcsb.org/download/{pdb_id}.pdb`) and save it locally.
2. For a given predicted complex, extract chain A's sequence and residue objects, and do the same for the reference structure's target chain. Use `Bio.Align.PairwiseAligner` (global alignment) to align the two sequences and identify which *residues* correspond to each other (skipping alignment gaps).
3. Collect the matched Cα atom pairs and use `Bio.PDB.Superimposer` to compute the rotation/translation that best fits the predicted target chain onto the reference target chain.
4. Apply that transform to the *predicted peptide chain's* atoms (not just the target chain), so the predicted peptide ends up in the reference structure's coordinate frame.
5. Compute an **overlap metric**: e.g. the fraction of transformed predicted-peptide Cα atoms that fall within some generous cutoff (start with ~8 Å, and justify whatever you settle on) of *any* Cα atom belonging to the reference structure's real peptide chain.

Report this overlap metric for every (target, real peptide) complex — you can skip computing it for the scrambled controls, or compute it anyway as a further sanity check (do scrambled peptides ever land in a real pocket by chance?).

In [ ]:
import requests
import numpy as np
import pandas as pd

from Bio.PDB import PDBParser, Superimposer
from Bio.Align import PairwiseAligner
from Bio.PDB.Polypeptide import three_to_index, index_to_one


def download_reference_structure(
    pdb_id: str,
    out_dir: Path
) -> Path:
    """
    Download the original PDB coordinate file for pdb_id,
    cache it in out_dir, and return the local path.
    """

    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{pdb_id}.pdb"

    if not out_path.exists():
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

        response = requests.get(url)
        response.raise_for_status()

        out_path.write_text(response.text)

    return out_path


def chain_sequence_and_residues(
    structure,
    chain_id: str
):
    """
    Return sequence and residue objects for one chain,
    skipping heteroatoms and waters.
    """

    chain = structure[0][chain_id]

    sequence = []
    residues = []

    for residue in chain:

        if residue.id[0] != " ":
            continue

        try:
            aa = index_to_one(
                three_to_index(residue.resname)
            )
        except KeyError:
            continue

        sequence.append(aa)
        residues.append(residue)

    return "".join(sequence), residues


def matched_ca_pairs(
    seq_a,
    residues_a,
    seq_b,
    residues_b
):
    """
    Globally align the two sequences and return matched
    CA atom pairs.
    """

    aligner = PairwiseAligner()
    aligner.mode = "global"

    alignment = aligner.align(
        seq_a,
        seq_b
    )[0]

    aligned_a, aligned_b = alignment.aligned

    ca_pairs = []

    for block_a, block_b in zip(
        aligned_a,
        aligned_b
    ):

        start_a, end_a = block_a
        start_b, end_b = block_b

        for i_a, i_b in zip(
            range(start_a, end_a),
            range(start_b, end_b)
        ):

            residue_a = residues_a[i_a]
            residue_b = residues_b[i_b]

            if (
                "CA" in residue_a
                and "CA" in residue_b
            ):
                ca_pairs.append(
                    (
                        residue_a["CA"],
                        residue_b["CA"]
                    )
                )

    return ca_pairs


def compute_pose_overlap(
    predicted_pdb_path: str,
    target_name: str,
    cutoff: float = 8.0
) -> dict:
    """
    Superimpose the predicted target onto the reference target
    and measure peptide-pose overlap.
    """

    reference_dir = WORKDIR / "reference_structures"

    pdb_id = TARGETS[target_name]["pdb_id"]

    reference_path = download_reference_structure(
        pdb_id,
        reference_dir
    )

    parser = PDBParser(QUIET=True)

    predicted_structure = parser.get_structure(
        "predicted",
        predicted_pdb_path
    )

    reference_structure = parser.get_structure(
        "reference",
        reference_path
    )

    # Predicted target = chain A
    pred_seq, pred_residues = chain_sequence_and_residues(
        predicted_structure,
        "A"
    )

    # Reference target chain
    ref_target_chain = TARGETS[target_name]["target_chain"]

    ref_seq, ref_residues = chain_sequence_and_residues(
        reference_structure,
        ref_target_chain
    )

    ca_pairs = matched_ca_pairs(
        pred_seq,
        pred_residues,
        ref_seq,
        ref_residues
    )

    n_matched = len(ca_pairs)

    if n_matched < 3:
        return {
            "n_matched_target_residues": n_matched,
            "overlap_fraction": None,
        }

    predicted_ca = [
        pair[0]
        for pair in ca_pairs
    ]

    reference_ca = [
        pair[1]
        for pair in ca_pairs
    ]

    superimposer = Superimposer()

    superimposer.set_atoms(
        reference_ca,
        predicted_ca
    )

    # Predicted peptide = chain B
    predicted_peptide_chain = predicted_structure[0]["B"]

    predicted_peptide_ca = [
        residue["CA"]
        for residue in predicted_peptide_chain
        if residue.id[0] == " "
        and "CA" in residue
    ]

    if len(predicted_peptide_ca) == 0:
        return {
            "n_matched_target_residues": n_matched,
            "overlap_fraction": None,
        }

    # Apply target-derived transformation to predicted peptide
    superimposer.apply(predicted_peptide_ca)

    # Reference real peptide
    ref_peptide_chain_id = TARGETS[
        target_name
    ]["reference_peptide_chain"]

    reference_peptide_chain = reference_structure[0][
        ref_peptide_chain_id
    ]

    reference_peptide_ca = [
        residue["CA"]
        for residue in reference_peptide_chain
        if residue.id[0] == " "
        and "CA" in residue
    ]

    if len(reference_peptide_ca) == 0:
        return {
            "n_matched_target_residues": n_matched,
            "overlap_fraction": None,
        }

    within_cutoff = 0

    for pred_atom in predicted_peptide_ca:

        distances = [
            np.linalg.norm(
                pred_atom.coord - ref_atom.coord
            )
            for ref_atom in reference_peptide_ca
        ]

        if min(distances) <= cutoff:
            within_cutoff += 1

    overlap_fraction = (
        within_cutoff
        / len(predicted_peptide_ca)
    )

    return {
        "n_matched_target_residues": n_matched,
        "overlap_fraction": overlap_fraction,
    }


overlap_rows = []


# Compute overlap only for PepMLM-generated peptides

for _, row in results_df[
    results_df["source"] == "pepmlm"
].iterrows():

    try:
        metrics = compute_pose_overlap(
            predicted_pdb_path=row["pdb_path"],
            target_name=row["target"],
        )

        overlap_rows.append({
            "target": row["target"],
            "peptide_id": row["peptide_id"],
            **metrics,
        })

    except Exception as e:
        print(
            f"Failed overlap for "
            f"{row['target']} / {row['peptide_id']}: {e}"
        )


overlap_df = pd.DataFrame(overlap_rows)

results_df = results_df.merge(
    overlap_df,
    on=["target", "peptide_id"],
    how="left"
)

results_df.to_csv(
    WORKDIR / "boltz_results_with_overlap.csv",
    index=False
)

## Step 8 — Statistics and a Justified Composite Score

Now put the pieces together with actual statistical reasoning instead of eyeballing a sorted column:

1. **Real vs. scrambled, per target.** For each target, compare the `iptm` distribution of `source == "pepmlm"` peptides against `source == "scrambled"` peptides using a **Mann–Whitney U test** (`scipy.stats.mannwhitneyu`) — appropriate here since you have small sample sizes and no reason to assume normality. Report the U statistic and p-value per target. What does it mean for your pipeline if a target shows *no* significant difference?
2. **Generation confidence vs. structural confidence.** Across all `"pepmlm"` rows (pooled or per-target — justify your choice), compute the **Spearman correlation** (`scipy.stats.spearmanr`) between `avg_log_prob` and `iptm`, with its p-value.
3. **Your composite ranking score.** Design a single score per real peptide that combines at least three of: `iptm`, `interface_contacts` (reuse the contact-counting function from the intro version of this exercise, or reimplement it), `overlap_fraction`, `avg_log_prob`. Standardize each component (e.g. z-score within its target group, since raw scales differ) before combining, and pick weights you can defend. Rank peptides per target by this score and save the final table.

There is no single correct composite score — the requirement is that your report explains and defends the choice you made.

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu, spearmanr
from Bio.PDB import PDBParser, NeighborSearch


def count_interface_contacts(
    pdb_path: str,
    chain1="A",
    chain2="B",
    cutoff=5.0
) -> int:
    """
    Count heavy-atom contacts between chain1 and chain2
    within cutoff distance.
    """

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("complex", pdb_path)

    model = structure[0]

    atoms_chain1 = [
        atom
        for atom in model[chain1].get_atoms()
        if atom.element != "H"
    ]

    atoms_chain2 = [
        atom
        for atom in model[chain2].get_atoms()
        if atom.element != "H"
    ]

    neighbor_search = NeighborSearch(atoms_chain2)

    contact_count = 0

    for atom in atoms_chain1:
        neighbors = neighbor_search.search(
            atom.coord,
            cutoff,
            level="A"
        )

        contact_count += len(neighbors)

    return contact_count


# Apply contact counting to every row

results_df["interface_contacts"] = results_df["pdb_path"].apply(
    count_interface_contacts
)


# ============================================================
# TODO 1: Mann-Whitney U test per target
# ============================================================

mann_whitney_rows = []

for target_name, group in results_df.groupby("target"):

    pepmlm_values = (
        group.loc[
            group["source"] == "pepmlm",
            "iptm"
        ]
        .dropna()
        .values
    )

    scrambled_values = (
        group.loc[
            group["source"] == "scrambled",
            "iptm"
        ]
        .dropna()
        .values
    )

    if len(pepmlm_values) > 0 and len(scrambled_values) > 0:

        test = mannwhitneyu(
            pepmlm_values,
            scrambled_values,
            alternative="two-sided"
        )

        mann_whitney_rows.append({
            "target": target_name,
            "U_statistic": test.statistic,
            "p_value": test.pvalue,
        })


mann_whitney_df = pd.DataFrame(mann_whitney_rows)

mann_whitney_df

In [ ]:
# ============================================================
# TODO 2: Spearman correlation
# ============================================================

pepmlm_rows = (
    results_df[
        results_df["source"] == "pepmlm"
    ]
    .dropna(
        subset=[
            "avg_log_prob",
            "iptm"
        ]
    )
)

spearman_result = spearmanr(
    pepmlm_rows["avg_log_prob"],
    pepmlm_rows["iptm"]
)

print(
    "Spearman rho:",
    spearman_result.statistic
)

print(
    "p-value:",
    spearman_result.pvalue
)

print(
    "n:",
    len(pepmlm_rows)
)

In [ ]:
# ============================================================
# TODO 3: Composite score
# ============================================================

ranked_df = results_df[
    results_df["source"] == "pepmlm"
].copy()


def zscore_within_target(series):

    std = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (
        series - series.mean()
    ) / std


metrics = [
    "iptm",
    "interface_contacts",
    "overlap_fraction",
    "avg_log_prob",
]


for metric in metrics:

    ranked_df[f"z_{metric}"] = (
        ranked_df
        .groupby("target")[metric]
        .transform(zscore_within_target)
    )


ranked_df["composite_score"] = (
    0.35 * ranked_df["z_iptm"]
    + 0.30 * ranked_df["z_overlap_fraction"]
    + 0.20 * ranked_df["z_interface_contacts"]
    + 0.15 * ranked_df["z_avg_log_prob"]
)


ranked_df["rank"] = (
    ranked_df
    .groupby("target")["composite_score"]
    .rank(
        ascending=False,
        method="min"
    )
)


ranked_df = ranked_df.sort_values(
    ["target", "rank"]
)


ranked_df.to_csv(
    WORKDIR / "final_ranked_results.csv",
    index=False
)


ranked_df[
    [
        "target",
        "peptide_id",
        "sequence",
        "iptm",
        "interface_contacts",
        "overlap_fraction",
        "avg_log_prob",
        "composite_score",
        "rank",
    ]
]

## Step 9 — Write the Report

Now the report has to actually defend a conclusion using evidence, not just present a table:

1. **Methods.** Targets (including how you found the third one and what search criteria you used), PepMLM decoding strategy and generation-confidence definition, scrambled-control construction, Boltz-1 settings, your composite score's components and weights.
2. **Results.** Final ranked table, the Mann–Whitney results per target, the Spearman correlation, at least one figure (e.g. `iptm`: real vs. scrambled, side by side per target; or `overlap_fraction` vs. `composite_score`).
3. **Discussion — answer these explicitly:**
   - For each target, was `iptm` for real peptides *statistically distinguishable* from scrambled controls? If not, what does that imply about trusting `iptm` alone as a ranking signal for that target?
   - Did generation confidence (`avg_log_prob`) predict structural confidence (`iptm`)? Was the correlation significant? What would a *strong* correlation actually tell you (and what would it not tell you)?
   - For your single best-ranked real peptide per target: what was its `overlap_fraction` with the true crystallographic peptide pose? Does a high composite score correspond to a pose that plausibly recapitulates the real binding mode, or could you have a high `iptm`/contact count with a *wrong* pose?
   - What are the specific ways your composite score could be gamed or misleading (e.g. a component that's easy to inflate without real improvement)? Would you change your weights after seeing your own results?
4. **Conclusion.** Which target/peptide combination would you actually recommend for follow-up, and what evidence specifically supports that (not just "highest score")?